# TurboQuant — Extreme KV Cache Compression

**Paper:** TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate (ICLR 2026)
**Authors:** Google Research
**Published:** March 2026

## References
- Paper: https://arxiv.org/abs/2504.19874
- Google Blog: https://research.google/blog/turboquant-redefining-ai-efficiency-with-extreme-compression/
- ICLR PDF: https://openreview.net/pdf/6593f484501e295cdbe7efcbc46d7f20fc7e741f.pdf
- PyTorch implementation: https://github.com/tonbistudio/turboquant-pytorch
- Community implementations: https://github.com/hackimov/turboquant-kv, https://github.com/0xSero/turboquant
- llama.cpp discussion: https://github.com/ggml-org/llama.cpp/discussions/20969
- Triton kernel blog: https://dejan.ai/blog/turboquant/
- Medium walkthrough: https://medium.com/data-science-collective/from-whiteboard-to-ide-implementing-googles-turboquant-kv-cache-compression-in-python-0e02b53a4640

## Prerequisites
Read `04-kv-cache.ipynb` first — TurboQuant directly extends the KV cache concept.

## What problem does this solve?

You already know from `04-kv-cache.ipynb` that KV cache grows linearly with sequence length.
For LLaMA-2-70B with 4096 context and batch=1:

```
KV cache = 2 × 64 heads × 128 d_k × 4096 tokens × float16
         = ~128MB per layer × 80 layers = 10GB just for cache
```

TurboQuant compresses each K and V vector from 16-bit to ~3-bit with **mathematically zero accuracy loss**.
6x less memory, up to 8x faster attention on H100.

## The Two-Stage Algorithm

### Stage 1: PolarQuant (MSE-optimal)

```
Input vector x ∈ ℝᵈ
    ↓
Rotate: y = Π @ x   (Π = random orthogonal matrix)
    ↓
After rotation, coordinates become nearly independent (Beta distribution)
    ↓
Apply Lloyd-Max quantization per coordinate (optimal scalar quantizer)
    ↓
Store: quantized indices (b-1 bits per coordinate)
```

**Why rotate first?** Raw embedding coordinates are correlated and have uneven magnitudes.
After random rotation, they spread evenly — all coordinates look the same statistically.
This means you can apply the same per-coordinate quantizer to every dimension optimally.

### Stage 2: QJL (Quantized Johnson-Lindenstrauss)

For **keys specifically**, we need to compute inner products (attention scores: Q @ K.T).
MSE reconstruction minimizes L2 error, but that doesn't guarantee accurate dot products.

```
Residual r = x - x̂   (what MSE quantizer missed)
    ↓
Project: Sr ∈ ℝᵐ   (S = random Gaussian matrix)
    ↓
Store only the signs: sign(Sr)  (1 bit per dimension)
    ↓
At query time: estimate inner product as:
    <q, x> ≈ <q, x̂_mse> + ‖r‖ × √(π/2)/m × <Sq, sign(Sr)>
```

**Community finding (from 8+ independent implementations):** MSE-only actually outperforms MSE+QJL for softmax attention in practice. Softmax amplifies QJL's noise variance. The V3 improvement uses all bits for MSE and drops QJL.

## Lloyd-Max Quantizer — the optimal scalar quantizer

Lloyd-Max finds the optimal centroids for a given distribution to minimize MSE.
After rotation, coordinates follow a Beta distribution, so we can precompute optimal centroids.

In [ ]:
import torch
import torch.nn as nn
import math
from typing import Optional, Tuple


class LloydMaxCodebook:
    """
    Precomputes optimal quantization centroids for a standard normal distribution
    using the Lloyd-Max algorithm (iterative optimization).
    After random rotation, each coordinate is approximately N(0,1).
    """
    def __init__(self, d: int, bits: int, n_iter: int = 100):
        self.n_levels = 2 ** bits
        self.centroids, self.boundaries = self._solve(n_iter)

    def _solve(self, n_iter: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Lloyd-Max iterative algorithm for N(0,1) distribution."""
        n = self.n_levels
        # Initialize boundaries uniformly
        boundaries = torch.linspace(-3.0, 3.0, n + 1)

        for _ in range(n_iter):
            # Update centroids: mean of Gaussian truncated to each interval
            centroids = torch.zeros(n)
            for i in range(n):
                a, b = boundaries[i], boundaries[i + 1]
                # E[X | a < X < b] for X ~ N(0,1)
                phi_a = torch.exp(-a**2 / 2) / math.sqrt(2 * math.pi)
                phi_b = torch.exp(-b**2 / 2) / math.sqrt(2 * math.pi)
                Phi_a = 0.5 * (1 + torch.erf(a / math.sqrt(2)))
                Phi_b = 0.5 * (1 + torch.erf(b / math.sqrt(2)))
                denom = Phi_b - Phi_a
                if denom < 1e-10:
                    centroids[i] = (a + b) / 2
                else:
                    centroids[i] = (phi_a - phi_b) / denom

            # Update boundaries: midpoints between centroids
            new_boundaries = torch.zeros(n + 1)
            new_boundaries[0]  = -float('inf')
            new_boundaries[-1] = float('inf')
            for i in range(1, n):
                new_boundaries[i] = (centroids[i-1] + centroids[i]) / 2
            boundaries = new_boundaries

        return centroids, boundaries


# Test: 4-bit Lloyd-Max gives 16 optimal centroids for N(0,1)
codebook = LloydMaxCodebook(d=64, bits=4)
print(f'4-bit centroids ({len(codebook.centroids)} levels):', codebook.centroids[:8], '...')

## Stage 1: TurboQuantMSE — rotation + Lloyd-Max quantization

In [ ]:
def generate_rotation_matrix(d: int, seed: int = 42, device: str = 'cpu') -> torch.Tensor:
    """
    Generate a random orthogonal rotation matrix via QR decomposition.
    Haar-distributed — uniform over all rotation matrices.
    Same matrix must be used at quantize AND dequantize time.
    (stored, not regenerated per call — hence the seed)
    """
    gen = torch.Generator(device='cpu')
    gen.manual_seed(seed)
    G = torch.randn(d, d, generator=gen)
    Q, R = torch.linalg.qr(G)
    # Fix sign ambiguity: ensure det(Q) = +1
    diag_sign = torch.sign(torch.diag(R))
    diag_sign[diag_sign == 0] = 1.0
    Q = Q * diag_sign.unsqueeze(0)
    return Q.to(device)


class TurboQuantMSE(nn.Module):
    """
    Stage 1 only: MSE-optimal quantizer.
    rotate → per-coordinate Lloyd-Max quantize → unrotate
    Used for VALUE vectors (need reconstruction, not inner products).
    """
    def __init__(self, d: int, bits: int, seed: int = 42, device: str = 'cpu'):
        super().__init__()
        self.d    = d
        self.bits = bits

        self.register_buffer('Pi', generate_rotation_matrix(d, seed=seed, device=device))

        cb = LloydMaxCodebook(d, bits)
        self.register_buffer('centroids', cb.centroids.to(device))

    def rotate(self, x):   return x @ self.Pi.T
    def unrotate(self, y): return y @ self.Pi

    def quantize(self, x: torch.Tensor) -> torch.Tensor:
        y      = self.rotate(x)                               # [batch, d]
        diffs  = y.unsqueeze(-1) - self.centroids             # [batch, d, n_levels]
        return diffs.abs().argmin(dim=-1)                     # [batch, d] int indices

    def dequantize(self, indices: torch.Tensor) -> torch.Tensor:
        y_hat = self.centroids[indices]                       # [batch, d]
        return self.unrotate(y_hat)

    def forward(self, x):
        idx   = self.quantize(x)
        x_hat = self.dequantize(idx)
        return x_hat, idx


# Test
d   = 64
mse = TurboQuantMSE(d=d, bits=4)
x   = torch.randn(10, d)

x_hat, idx = mse(x)
mse_error  = ((x - x_hat) ** 2).mean().item()

print(f'Input shape:    {x.shape}')
print(f'Indices shape:  {idx.shape}   (stores {idx.numel() * 4} bits = {idx.numel() * 4 / 8:.0f} bytes)')
print(f'MSE error:      {mse_error:.6f}')
print(f'Cosine sim avg: {torch.nn.functional.cosine_similarity(x, x_hat).mean().item():.4f}')

## Stage 1 + 2: TurboQuantProd — for KEY vectors

Keys need accurate inner products (attention scores: Q @ K.T).
Add QJL residual correction on top of MSE.

In [ ]:
def generate_qjl_matrix(d: int, m: int, seed: int = 43, device: str = 'cpu') -> torch.Tensor:
    gen = torch.Generator(device='cpu')
    gen.manual_seed(seed)
    return torch.randn(m, d, generator=gen).to(device)


class TurboQuantProd(nn.Module):
    """
    Stage 1 (b-1 bit MSE) + Stage 2 (1-bit QJL on residual).
    Total: b bits per coordinate.
    Used for KEY vectors — need unbiased inner product estimates.
    """
    def __init__(self, d: int, bits: int, seed: int = 42, device: str = 'cpu'):
        super().__init__()
        self.d        = d
        self.bits     = bits
        self.mse_bits = max(bits - 1, 1)

        self.mse = TurboQuantMSE(d, self.mse_bits, seed=seed, device=device)
        self.register_buffer('S', generate_qjl_matrix(d, m=d, seed=seed+1, device=device))

    def quantize(self, x: torch.Tensor) -> dict:
        x_hat, mse_idx  = self.mse(x)
        residual        = x - x_hat
        residual_norm   = torch.norm(residual, dim=-1)       # [batch]

        projected  = residual @ self.S.T                     # [batch, d]
        qjl_signs  = torch.sign(projected)
        qjl_signs[qjl_signs == 0] = 1.0

        return {'mse_indices': mse_idx, 'qjl_signs': qjl_signs, 'residual_norm': residual_norm}

    def inner_product(self, y: torch.Tensor, compressed: dict) -> torch.Tensor:
        """
        Unbiased estimate of <y, x> without decompressing x.
        Estimator: <y, x_mse> + ‖r‖ × √(π/2)/m × <Sy, sign(Sr)>
        """
        x_mse  = self.mse.dequantize(compressed['mse_indices'])
        term1  = (y * x_mse).sum(dim=-1)

        y_proj = y @ self.S.T
        qjl_ip = (y_proj * compressed['qjl_signs']).sum(dim=-1)
        term2  = compressed['residual_norm'] * math.sqrt(math.pi / 2) / self.d * qjl_ip

        return term1 + term2


# Test inner product accuracy
d    = 64
prod = TurboQuantProd(d=d, bits=4)

keys    = torch.randn(20, d)
queries = torch.randn(5, d)

compressed = prod.quantize(keys)

# True inner products
true_ip = queries @ keys.T   # [5, 20]

# Estimated inner products via TurboQuant
est_ip  = torch.stack([prod.inner_product(queries[i].unsqueeze(0).expand(20, -1), compressed)
                       for i in range(5)])

err = (true_ip - est_ip).abs().mean().item()
print(f'Mean absolute error in attention scores: {err:.6f}')
print(f'Relative error: {(err / true_ip.abs().mean()).item():.4f}')

## TurboQuantKVCache — full KV cache with compression

Keys → TurboQuantProd (inner product accuracy)
Values → TurboQuantMSE (reconstruction accuracy)

In [ ]:
class TurboQuantKVCache:
    """
    Drop-in KV cache replacement using TurboQuant compression.
    Keys compressed with TurboQuantProd (for attention dot products).
    Values compressed with TurboQuantMSE (for weighted sum reconstruction).
    """
    def __init__(self, d_key: int, d_val: int, bits: int = 3, seed: int = 42, device: str = 'cpu'):
        self.key_q = TurboQuantProd(d_key, bits, seed=seed, device=device)
        self.val_q = TurboQuantMSE(d_val, bits,  seed=seed+100, device=device)
        self.key_cache = []
        self.val_cache = []

    def append(self, keys: torch.Tensor, values: torch.Tensor):
        """keys, values: [seq_len, d]"""
        self.key_cache.append(self.key_q.quantize(keys))
        self.val_cache.append({'indices': self.val_q.quantize(values), 'shape': values.shape})

    def attend(self, query: torch.Tensor, scale: float) -> torch.Tensor:
        """
        Full attention with compressed KV cache.
        query: [d_key]
        Returns: attention output [d_val]
        """
        # Compute scores against all cached keys
        scores = []
        for ck in self.key_cache:
            q_exp = query.unsqueeze(0).expand(ck['mse_indices'].shape[0], -1)
            s = self.key_q.inner_product(q_exp, ck)
            scores.append(s)
        scores = torch.cat(scores) * scale             # [total_seq_len]

        # Softmax
        weights = torch.softmax(scores, dim=0)         # [total_seq_len]

        # Weighted sum of reconstructed values
        all_values = torch.cat([self.val_q.dequantize(cv['indices']) for cv in self.val_cache])
        return (weights.unsqueeze(-1) * all_values).sum(0)  # [d_val]

    def memory_bits(self) -> dict:
        n_k     = sum(c['mse_indices'].numel() for c in self.key_cache)
        n_qjl   = sum(c['qjl_signs'].numel()   for c in self.key_cache)
        n_norms = sum(c['residual_norm'].numel()for c in self.key_cache)
        n_v     = sum(c['indices'].numel()      for c in self.val_cache)

        compressed_bits = n_k * self.key_q.mse_bits + n_qjl * 1 + n_norms * 16 + n_v * self.val_q.bits
        fp16_bits       = (n_k + n_v) * 16
        ratio           = fp16_bits / compressed_bits if compressed_bits > 0 else 0
        return {'compressed_bits': compressed_bits, 'fp16_bits': fp16_bits, 'compression_ratio': ratio}


# Test
d_key, d_val = 64, 64
cache = TurboQuantKVCache(d_key=d_key, d_val=d_val, bits=3)

# Simulate 100 tokens being cached
for _ in range(10):
    k = torch.randn(10, d_key)
    v = torch.randn(10, d_val)
    cache.append(k, v)

# Attend with a query
query    = torch.randn(d_key)
out      = cache.attend(query, scale=1.0/math.sqrt(d_key))
mem      = cache.memory_bits()

print(f'Attention output shape: {out.shape}')
print(f'Cached tokens:          100')
print(f'Compressed bits:        {mem["compressed_bits"]:,}')
print(f'FP16 would need:        {mem["fp16_bits"]:,} bits')
print(f'Compression ratio:      {mem["compression_ratio"]:.2f}x')

## Memory savings at scale

In [ ]:
print('KV Cache memory at different sequence lengths (LLaMA-2 style, d=128):\n')
print(f'{"Seq len":>10} | {"FP16 MB":>10} | {"TurboQ 3-bit MB":>16} | {"Ratio":>8}')
print('-' * 55)

for seq_len in [1024, 4096, 16384, 32768, 65536]:
    d         = 128
    n_heads   = 32
    n_layers  = 32

    fp16_mb   = 2 * n_heads * d * seq_len * n_layers * 2 / 1e6  # 2=K+V, 2=bytes fp16
    turbo_mb  = 2 * n_heads * d * seq_len * n_layers * 3 / 8 / 1e6  # 3 bits
    ratio     = fp16_mb / turbo_mb

    print(f'{seq_len:>10,} | {fp16_mb:>10.1f} | {turbo_mb:>16.1f} | {ratio:>7.1f}x')

## Community finding: MSE-only (V3) beats MSE+QJL in practice

From [tonbistudio/turboquant-pytorch](https://github.com/tonbistudio/turboquant-pytorch) and 6+ other implementations:

> *Six independent teams confirmed that MSE-only outperforms MSE+QJL for softmax-based attention, as softmax amplifies QJL's random noise variance.*

**V3 improvements:**
- Drop QJL, use all bits for MSE
- Asymmetric allocation: Keys get more bits than Values (keys matter more for attention patterns)
- Layer-adaptive: protect early/late layers with extra bits
- Bit-packed storage for actual compression

```python
# V3 config that achieves 2x compression with exact generation:
# Keys: 4-bit MSE, Values: 2-bit MSE
# + 128-token residual window (last N tokens stored in FP16, rest compressed)
```

This is still very new (paper: March 2026). The community is actively exploring what config works best per model architecture.

## Where TurboQuant fits in your post-training platform

```
Your platform flow:

Fine-tuned model (SFT/DPO/GRPO from Level 5)
    ↓ quantize weights (GPTQ/AWQ — weight quantization)
  Model weights in 4-bit
    ↓ serve
Inference engine (vLLM / llama.cpp)
    ↓ during generation
  KV Cache grows with sequence length
    ↓ TurboQuant compresses the KV cache ← THIS is where TurboQuant plugs in
  6x less KV cache memory → serve longer contexts or more users
```

**TurboQuant is inference-time, not training-time.** It sits in the serving layer, not in fine-tuning.